In [ ]:
# Databricks notebook source
# tutorial_name: 02-Day8-Medallion-MultiTask-Workflow
# notebookName: 02-Day8-Medallion-MultiTask-Workflow
# COMMAND ----------

# DBTITLE 1,Paths (same pattern as Days 5–7)
notepoint_rel = "hands-on/day-08/notebooks/02-Day8-Medallion-MultiTask-Workflow.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
SOURCE_CSV = BASE_PATH + "/2010-summary.csv"
SOURCE_JSON = BASE_PATH + "/json/2015-summary.json"
DAY8_ROOT = f"{BASE_PATH}day08-{STUDENT_ID}"
MED_ROOT = f"{DAY8_ROOT}/medallion"
BRONZE_PATH = f"{MED_ROOT}/bronze_flight_routes"
SILVER_PATH = f"{MED_ROOT}/silver_flight_routes"
GOLD_PATH = f"{MED_ROOT}/gold_by_destination"
METRICS_PATH = f"{DAY8_ROOT}/workflow_metrics/run_metrics"
# COMMAND ----------

# DBTITLE 1,Prerequisite check
print("DAY8_ROOT:", DAY8_ROOT)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("✓ P_BASIC readable")
except Exception as e:
    print(f"✗ P_BASIC: {e}")
try:
    spark.read.format("csv").option("header", True).load(SOURCE_CSV).limit(1).collect()
    print("✓ SOURCE_CSV readable")
except Exception as e:
    print(f"✗ SOURCE_CSV: {e}")
try:
    spark.read.format("json").load(SOURCE_JSON).limit(1).collect()
    print("✓ SOURCE_JSON readable (optional depth)")
except Exception as e:
    print("(optional) SOURCE_JSON:", type(e).__name__)
# COMMAND ----------


# Day 8 — Item 22: Workflow orchestration & production-style pipeline

Syllabus item 22 — multi-task jobs, **task dependencies**, **error handling**, **scheduling**; hands-on **Bronze → Silver → Gold**.

**Paths:** All writes stay under **`day08-{STUDENT_ID}/`** — same discipline as **`day06-{STUDENT_ID}`** and **`day07-{STUDENT_ID}`**.

**Parameterization:** Widget **`pipeline_stage`** drives behavior so one notebook can back **three job tasks** (`bronze`, `silver`, `gold`) with **Depends on** edges.


## Figure — one notebook, three job tasks (item 22)

```mermaid
flowchart LR
  subgraph job[Single Job]
    B[Task bronze<br/>pipeline_stage=bronze]
    S[Task silver<br/>pipeline_stage=silver]
    G[Task gold<br/>pipeline_stage=gold]
  end
  B --> S
  S --> G
  B -.-> NB[Same notebook file]
  S -.-> NB
  G -.-> NB
```

**Why one notebook?** Same code, different **widget / parameter** per task — easy to maintain in training.


## Prerequisites chain (earlier days)

| Day | What you reuse |
|-----|----------------|
| **3** | Medallion **language** (bronze / silver / gold) |
| **4** | If using UC tables instead of paths, swap reads after instructor approval |
| **5** | **`SOURCE_CSV`**, **`P_BASIC`**, Delta on ABFS |
| **6–7** | Delta history, streaming (not required here) |

If **`P_BASIC`** or **`2010-summary.csv`** fails the check above, fix Day 5 data landing first.


## Medallion flow (today)

```mermaid
flowchart LR
  CSV[2010-summary.csv] --> B[Bronze Delta]
  B --> S[Silver Delta]
  S --> G[Gold Delta]
```

- **Bronze:** raw-aligned landing (schema from CSV, **overwrite**).
- **Silver:** **validated** rows (non-null keys), **overwrite**.
- **Gold:** **aggregated** KPI by destination, **overwrite**.


## Figure — medallion + quality gate (detailed)

```mermaid
flowchart TB
  CSV[2010-summary.csv on ABFS]
  CSV --> BR[Bronze Delta<br/>raw + ingest_ts]
  BR --> Q{Keys non-null?}
  Q -->|filter| SV[Silver Delta<br/>silver_built_ts]
  SV --> AG[Aggregate by DEST]
  AG --> GD[Gold Delta<br/>total_flights]
```

**ASCII stack:**

```text
  CSV
   ↓  overwrite
  Bronze (wide, raw-ish)
   ↓  filter null keys → overwrite
  Silver (clean keys)
   ↓  groupBy dest, sum(count) → overwrite
  Gold (KPI by destination)
```


## Lab 1 — Inspect source file (Bronze input)

Before writing Bronze, confirm **schema** and **row count** match expectations (same file as Day 5 **01** notebook).


In [ ]:
src_preview = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(SOURCE_CSV)
)
print("Columns:", src_preview.columns)
print("Row count:", src_preview.count())
src_preview.show(5, truncate=False)


In [ ]:
# Exercise: infer schema without writing Delta (ties to Bronze design)
_cols = src_preview.dtypes
print('Bronze will store', len(_cols), 'columns:')
for name, typ in _cols:
    print(f'  - {name}: {typ}')


## Lab 2 — Bronze layer design

**Goals**

- Persist CSV to **Delta** for repeatable downstream reads.
- Add **`ingest_ts`** for audit (when this layer was built).

**Storage path:** `BRONZE_PATH` (printed in the first cell).


## Lab 3 — Silver layer rules

- Require **`DEST_COUNTRY_NAME`** and **`ORIGIN_COUNTRY_NAME`** **not null**.
- Optional: you could add **`count > 0`** — not enforced here to stay close to raw data.

**Input:** Bronze Delta. **Output:** Silver Delta (**overwrite**).


## Lab 4 — Gold layer KPI

- **Grain:** one row per **`DEST_COUNTRY_NAME`**.
- **Measure:** **`total_flights`** = `sum(count)` (same **`count`** column as Days 1–2 flights lab).

Downstream jobs might **publish** this table to a BI tool or a **Unity Catalog** table (out of scope unless instructor extends).


In [ ]:
dbutils.widgets.dropdown("pipeline_stage", "all", ["all", "bronze", "silver", "gold"])
stage = dbutils.widgets.get("pipeline_stage").strip().lower()
print("pipeline_stage =", stage)


## Figure — widget → branch (how the job parameter maps)

```text
  Job task parameter: pipeline_stage = silver
           |
  dbutils.widgets.get('pipeline_stage')  -->  silver
           |
  run_silver() only
```

Interactive **all** runs bronze then silver then gold in one session.


In [ ]:
# Exercise: preview which stages will run (after widget is defined)
_st = dbutils.widgets.get('pipeline_stage').strip().lower()
order = []
if _st == 'all':
    order = ['bronze', 'silver', 'gold']
elif _st in ('bronze', 'silver', 'gold'):
    order = [_st]
print('Stages to execute:', order)


## Lab 5 — Stage implementations (functions)

The next cell **defines** `run_bronze`, `run_silver`, `run_gold`. The cell **after** dispatches based on **`pipeline_stage`**.

**Jobs:** create **three tasks** with parameters **`bronze`**, **`silver`**, **`gold`** and dependency chain.


In [ ]:
from pyspark.sql import functions as F


def run_bronze():
    df = (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(SOURCE_CSV)
    )
    df = df.withColumn("ingest_ts", F.current_timestamp())
    df.write.format("delta").mode("overwrite").save(BRONZE_PATH)
    print("Bronze rows:", df.count(), "→", BRONZE_PATH)


def run_silver():
    df = spark.read.format("delta").load(BRONZE_PATH)
    out = df.filter(
        F.col("DEST_COUNTRY_NAME").isNotNull() & F.col("ORIGIN_COUNTRY_NAME").isNotNull()
    )
    out = out.withColumn("silver_built_ts", F.current_timestamp())
    out.write.format("delta").mode("overwrite").save(SILVER_PATH)
    print("Silver rows:", out.count(), "→", SILVER_PATH)


def run_gold():
    df = spark.read.format("delta").load(SILVER_PATH)
    out = df.groupBy("DEST_COUNTRY_NAME").agg(F.sum("count").alias("total_flights"))
    out = out.withColumn("gold_built_ts", F.current_timestamp())
    out.write.format("delta").mode("overwrite").save(GOLD_PATH)
    print("Gold rows:", out.count(), "→", GOLD_PATH)


In [ ]:
if stage == "all":
    run_bronze()
    run_silver()
    run_gold()
elif stage == "bronze":
    run_bronze()
elif stage == "silver":
    run_silver()
elif stage == "gold":
    run_gold()
else:
    raise ValueError(f"Unknown pipeline_stage: {stage}")

print("Dispatch finished for stage:", stage)


## Figure — dependency guarantees order

When the **Job** defines **silver depends on bronze**, the platform will not start **silver** until **bronze** succeeds — even if someone mis-orders parameters in the UI.

```mermaid
sequenceDiagram
  participant J as Job run
  participant B as Task bronze
  participant S as Task silver
  participant G as Task gold
  J->>B: start
  B-->>J: Succeeded
  J->>S: start
  S-->>J: Succeeded
  J->>G: start
  G-->>J: Succeeded
```


In [ ]:
# Exercise: quick read of each layer after dispatch (validates practical output)
for label, pth in [('bronze', BRONZE_PATH), ('silver', SILVER_PATH), ('gold', GOLD_PATH)]:
    try:
        spark.read.format('delta').load(pth).limit(3).show(3, truncate=False)
    except Exception as e:
        print(label, 'not available:', type(e).__name__)


## Lab 6 — Post-run validation (counts per layer)

Run after **`all`** or after the last stage in the chain.


In [ ]:
def _safe_count(path):
    try:
        return spark.read.format("delta").load(path).count()
    except Exception as e:
        return f"(missing: {e})"


print("Bronze count:", _safe_count(BRONZE_PATH))
print("Silver count:", _safe_count(SILVER_PATH))
print("Gold count:", _safe_count(GOLD_PATH))


## Lab 7 — Data quality on Bronze (nulls)

Reinforces **why** Silver filters keys.


In [ ]:
try:
    b = spark.read.format("delta").load(BRONZE_PATH)
    b.select(
        F.sum(F.when(F.col("DEST_COUNTRY_NAME").isNull(), 1).otherwise(0)).alias("null_dest"),
        F.sum(F.when(F.col("ORIGIN_COUNTRY_NAME").isNull(), 1).otherwise(0)).alias("null_origin"),
        F.count(F.lit(1)).alias("rows"),
    ).show()
except Exception as e:
    print("Run bronze stage first:", e)


## Lab 8 — Delta history (Bronze / Silver / Gold)

Same **`DESCRIBE HISTORY`** pattern as **Day 5** and **Day 6**.


In [ ]:
for label, pth in [
    ("bronze", BRONZE_PATH),
    ("silver", SILVER_PATH),
    ("gold", GOLD_PATH),
]:
    try:
        print("--- HISTORY:", label, "---")
        spark.sql(f"DESCRIBE HISTORY delta.`{pth}`").select("version", "operation").show(5, truncate=False)
    except Exception as e:
        print(label, "skip:", type(e).__name__)


## Lab 9 — Sample Gold output

Top destinations by **`total_flights`**.


In [ ]:
try:
    spark.read.format("delta").load(GOLD_PATH).orderBy(F.desc("total_flights")).show(15, truncate=False)
except Exception as e:
    print("Gold not ready:", e)


## Lab 10 — Sanity vs `P_BASIC` (optional cross-check)

Compare **row count** only (schemas differ). Large gaps may mean CSV vs Delta build differences — use for class discussion, not strict equality.


In [ ]:
try:
    n_basic = spark.read.format("delta").load(P_BASIC).count()
    n_bronze = spark.read.format("delta").load(BRONZE_PATH).count()
    print("P_BASIC rows:", n_basic)
    print("Bronze rows:", n_bronze)
except Exception as e:
    print("Skip compare:", e)


## Lab 11 — Append pipeline metrics row

Simulates a **metrics** table updated at end of a **gold** task (pattern for observability).


In [ ]:
try:
    gold_n = spark.read.format("delta").load(GOLD_PATH).count()
except Exception:
    gold_n = -1
metrics = spark.sql(
    f"SELECT '{STUDENT_ID}' AS student_id, '{stage}' AS last_stage, "
    f"current_timestamp() AS metric_ts, {int(gold_n)} AS gold_dim_rows"
)
metrics.write.format("delta").mode("append").save(METRICS_PATH)
spark.read.format("delta").load(METRICS_PATH).orderBy("metric_ts", ascending=False).show(5, truncate=False)


## Multi-task job configuration (reference)

| Task key | Depends on | Parameter `pipeline_stage` |
|----------|------------|----------------------------|
| `bronze` | — | `bronze` |
| `silver` | `bronze` | `silver` |
| `gold` | `silver` | `gold` |

**Error handling (UI):** per-task **timeout**, **max retries**, **email / Teams** on failure.  
**Scheduling:** pause in shared sandboxes when class ends.


## Idempotency note

Stages use **`overwrite`** on Delta paths. Re-running **bronze** replaces raw landing; **silver** and **gold** then rebuild.  
For **append-only** sources, switch to **`merge`** / **`append`** patterns (Day 5 **MERGE** lab) in a future iteration.


## Defensive run: Gold without Silver

If someone runs **`pipeline_stage=gold`** before silver exists, the dispatch cell **fails** — expected.  
In production, use **task dependencies** so **gold** never starts unless **silver** succeeded.


## Extended hands-on checklist (item 22)

1. Widget = **`all`** → run full pipeline interactively; pass Labs 6–11.
2. Create job **`day08_medallion_{STUDENT_ID}`** with **three** notebook tasks on **this file**.
3. Map parameters as in the table above; **Run now**; verify DAG order.
4. Induce a failure (e.g. temporary wrong path in a **clone** notebook) → watch **retry** / **notification**.
5. Export job **JSON** (API) for Git — tie-in to CI/CD extras in `databricks/**/15-*`.


In [ ]:
# Optional cleanup — uncomment only when you own this prefix
# dbutils.fs.rm(DAY8_ROOT, recurse=True)
